In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.tree


>DOM tree
output-file: core.dom.tree
title: core.dom.tree

This notebook provides utility functions for DOM tree manipulation
---

In [ ]:
# | default_exp core.dom.tree

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import json
import os
import re
from collections.abc import Callable
from enum import Enum
from functools import wraps
from inspect import Parameter, signature
from pathlib import Path
from typing import Optional

from pydantic import BaseModel, Field

from ollama import AsyncClient
from openai import AsyncOpenAI

import jsoncfg
from jsoncfg.config_classes import (
    ConfigJSONArray,
    ConfigJSONObject,
    ConfigJSONScalar,
    ConfigNode,
)

from ribosome.core.dom.utils import get_summary_response_async, get_image_summary_response_async, image_link_pattern,get_image_summary_async

In [ ]:
# | export
def map_tree(node, transform: Callable):
    """
    Recursively apply a transformation function to each node in a tree structure.

    Args:
        node: The root node of the tree, which can be a dict or a list.
        transform: A callable that takes a node and returns a transformed node.

    Returns:
        The transformed tree.
    """
    
    node = transform(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    map_tree(child, transform)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = map_tree(value, transform)
    elif isinstance(node, list):
        node = [
            map_tree(child, transform) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node

In [ ]:
# | export
async def map_tree_async(node, transform: Callable, on_exit: Optional[Callable] = None):
    """Recursively apply an asynchronous transformation function to each node in a tree structure.

    Args:
        node: The root node of the tree, which can be a dict or a list.
        transform: A callable that takes a node and returns a transformed node.
        on_exit: An optional callable that is called with the node after it has been transformed.

    Returns:
        The transformed tree.
    """
    
    node = await transform(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    await map_tree_async(child, transform, on_exit)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = await map_tree_async(value, transform, on_exit)
    elif isinstance(node, list):
        node = [
            await map_tree_async(child, transform, on_exit) if isinstance(child, (dict, list)) else child
            for child in node
        ]

    if on_exit is not None:
        await on_exit(node)
    return node


In [ ]:
# | export
def walk_node_obj(node, action:Optional[Callable]=None)->None:
    if action:
        node = action(node)
    if isinstance(node, ConfigJSONObject):
        # Dictionary
        for key, value in node:
            print(f"key   \"{key}\" at line {jsoncfg.node_location(value).line}")
            walk_node_obj(value)
    elif isinstance(node, ConfigJSONArray):
        # Array
        for item in node:
            walk_node_obj(item)
    elif isinstance(node, ConfigJSONScalar):
        # Scalar
        value = node()
        if isinstance(value, str):
            value = value.strip()
        print(f"value \"{value}\" at line {jsoncfg.node_location(node).line}")
    else:
        raise ValueError(f"Unknown node type: {type(node)}")

In [ ]:
# | export
def walk_nodes_with_line_number(raw_json: str, action: Optional[Callable] = None) -> None:
    """Walk the AST config structure and print node locations by source line.
    
    Args:
        action: Optional transform applied to each visited node.
    """
    if action is None:
        def _identity(node):
            return node
        action = _identity

    ast = jsoncfg.load_config(raw_json)


    ast = walk_node_obj(ast)


In [ ]:
# | export
def walk_node(node, action: Optional[Callable] = None):
    if action:
        node = action(node)
    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                node[key] = [
                    walk_node(child)
                    if isinstance(child, (dict, list))
                    else child
                    for child in value
                ]
            elif isinstance(value, dict):
                node[key] = walk_node(value)
    elif isinstance(node, list):
        node = [
            walk_node(child, action) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node

In [ ]:
# | export
def walk(raw_json:str, action: Optional[Callable] = None) -> str:
        """ Walks through the Markdown AST and applies an action to each node.
        If no action is provided, it defaults to the identity function.

        returns:
            str: The modified JSON string after applying the action to each node.
        """

        ast = json.loads(raw_json)
        ast = walk_node(ast, action)
        return json.dumps(ast, ensure_ascii=False).encode("utf-8").decode("utf-8")


In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()